# Assignment 3 — Milestone I: Natural Language Processing
## Task 1 — Basic Text Pre-processing

**Student Name:** Yoshita Sarin
**Student ID:** s4225113

### Environment
* Python 3 (Jupyter Notebook)

### Libraries used
| Library | What we use it for |
|---|---|
| `pandas` | Reading the review CSV file and saving the cleaned version back to disk |
| `numpy` | Calculating simple averages and standard deviations of review length |
| `nltk.RegexpTokenizer` | Splitting each review into individual words using the pattern given in the brief |
| `nltk.stem.WordNetLemmatizer` | Converting words to their base form (e.g. `products` → `product`) |
| `itertools.chain` | Joining many small lists of words into one big list |
| `collections.Counter` | Counting how often each word appears |


## Introduction

This notebook does **Task 1** of the assignment: cleaning up the cosmetics and beauty
product reviews so they are ready for machine-learning later on. The dataset has about
61,000 customer reviews, and we only work with the `review_text` column (the actual review
the customer wrote).

### What we are producing
By the end of this notebook we will have created two files:
1. **`processed.csv`**
2. **`vocab.txt`**

### Steps we will follow
We do the cleaning in this order. The order matters — for example, we lemmatise (turn
words into their base form) **before** counting word frequencies, so that `love` and
`loved` are counted as the same word.

| # | Step | What it does |
|---|---|---|
| 1 | Load the reviews | Read the CSV file into a table |
| 2 | Split each review into words | Use the pattern `r"[a-zA-Z]+(?:[-'][a-zA-Z]+)?"` |
| 3 | Make everything lowercase | So `Skin` and `skin` are treated the same |
| 4 | Remove very short words | Drop anything shorter than 2 letters |
| 5 | Remove common stop words | Use the supplied `stopwords_en.txt` file |
| 6 | Lemmatise the remaining words | Turn each word into its base form |
| 7 | Remove words that appear only once | These are usually typos or rare names |
| 8 | Remove the 20 most common words | They appear in nearly every review and aren't useful |
| 9 | Save `processed.csv` and `vocab.txt` | The two files the assignment asks for |


## Importing libraries


In [ ]:
# Standard Python libraries
from collections import Counter        # for counting how often each word appears
from itertools import chain            # for joining many small word-lists into one big list

# Data-science libraries
import numpy as np                     # for averages and standard deviations
import pandas as pd                    # for reading and writing the CSV file

# NLTK tools
import nltk
from nltk import RegexpTokenizer       # splits text into words using a pattern we give it
from nltk.stem import WordNetLemmatizer  # turns words into their base form

# Download the WordNet data the lemmatiser needs. quiet=True hides the progress message.
# These calls do nothing if the data is already on the computer.
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)


## 1.1 Loading and looking at the data

Before changing any text, let's check the file: how many reviews there are, what columns
are available, and whether any reviews are missing.


In [ ]:
# Read the CSV file. It must be in the same folder as this notebook.
raw_df = pd.read_csv('../data/cosmetics_beauty_products_reviews.csv')

print('Number of reviews:', len(raw_df))
print('Number of columns:', raw_df.shape[1])
print()
print('Columns in the file:')
for col in raw_df.columns:
    print(f'  - {col}')

# Show the first three rows so we can see what the data looks like.
raw_df.head(3)


**What we can see:**
* There are 15 columns. Task 1 only asks us to clean `review_text`, but we will keep all
  the other columns when we save the cleaned file so Task 3 can use them later.

In [ ]:
# A handful of reviews have no text at all (NaN). We replace those with empty strings
# so the splitter doesn't crash when it tries to process them.
n_missing = raw_df['review_text'].isna().sum()
print(f'Reviews with missing text: {n_missing}')

raw_df['review_text'] = raw_df['review_text'].fillna('').astype(str)

# Print one example so we can see what a raw review looks like.
print('\nExample raw review:')
print(raw_df['review_text'].iloc[0])


## 1.2 Cleaning the review text

We now do the cleaning steps one at a time. After each step we print some simple
statistics (number of unique words, number of total words, average review length, etc.)
so we can see exactly what each step changes.


### 1.2.1 A small helper for printing statistics

This little function takes the cleaned reviews and prints how many unique words there
are, how many words in total, and how long the reviews are on average. We will call it
after every cleaning step so the effect of each step is easy to see.


In [ ]:
def stats_print(tk_reviews):
    """Print simple statistics about the cleaned reviews.

    ``tk_reviews`` is a list where each item is itself a list of words for one review.
    """
    words = list(chain.from_iterable(tk_reviews))   # all the words from all reviews, in one big list
    unique_words = set(words)                       # the unique ones (a Python set has no duplicates)
    print('Number of unique words        :', len(unique_words))
    print('Total number of words         :', len(words))
    if words:
        # how varied the wording is - higher means more variety, lower means more repetition
        print('Word variety (unique/total)  :', round(len(unique_words) / len(words), 5))
    print('Number of reviews             :', len(tk_reviews))
    lens = [len(r) for r in tk_reviews]
    print('Average review length         :', round(float(np.mean(lens)), 2))
    print('Longest review (in words)     :', int(np.max(lens)))
    print('Shortest review (in words)    :', int(np.min(lens)))
    print('Standard deviation of length  :', round(float(np.std(lens)), 2))


### 1.3 Splitting reviews into words and making them lowercase (steps 2 & 3)

The brief tells us to use this exact pattern to split text into words:
`r"[a-zA-Z]+(?:[-'][a-zA-Z]+)?"`

In plain English, this pattern says:
* Match one or more letters in a row (e.g. `skin`, `lipstick`).
* Optionally allow **one** hyphen or apostrophe in the middle, followed by more letters
  (so `long-lasting` and `it's` are kept as single words).
* No numbers, no punctuation marks, no symbols.

We make the text lowercase **before** splitting. This way every word that comes out is
already in lowercase, so we don't have to do it as a separate step.


In [ ]:
# The pattern from the brief - we must use this exact one.
PATTERN = r"[a-zA-Z]+(?:[-'][a-zA-Z]+)?"
tokenizer = RegexpTokenizer(PATTERN)

def tokenise_review(text: str):
    """Make the text lowercase, then split it into words using the required pattern."""
    return tokenizer.tokenize(text.lower())

# Apply this to every review. ``tk_reviews`` ends up as a list with one entry per
# review, where each entry is itself a list of the words in that review.
tk_reviews = [tokenise_review(r) for r in raw_df['review_text'].tolist()]

# Show the result for the very first review so we can compare with the raw text above.
print('First review after splitting and lowercasing:')
print(tk_reviews[0])
print()
stats_print(tk_reviews)


**What this did:** Punctuation, numbers, currency symbols and emojis are simply ignored
by the pattern, which is exactly what we want. Right now the unique-word count is the
highest it will ever be — every following step removes words.


### 1.4 Removing very short words (step 4)

Words of just one letter (`a`, `i`, `u`, …) almost never carry useful meaning. They
usually come from typing shortcuts or leftovers from punctuation, so we drop them.


In [ ]:
# Keep only words that are 2 or more letters long.
tk_reviews = [[w for w in review if len(w) >= 2] for review in tk_reviews]

stats_print(tk_reviews)


### 1.5 Removing stop words (step 5)

Stop words are very common words like `the`, `is`, `and`, `a`, `of`. They appear in
almost every sentence and don't really tell us anything about whether a review is
positive or negative.

We use the **stop-word list provided with the assignment** (`stopwords_en.txt`)


In [ ]:
# Load the stop words into a Python set, which is fast for membership checks.
with open('../data/stopwords_en.txt', 'r', encoding='utf-8') as f:
    stopwords_en = {line.strip() for line in f if line.strip()}

print('Number of stop words loaded:', len(stopwords_en))
print('First 10 (alphabetical) :', sorted(stopwords_en)[:10])

# Keep only the words that are NOT in the stop-word list.
tk_reviews = [[w for w in review if w not in stopwords_en] for review in tk_reviews]

stats_print(tk_reviews)


**What this did:** This is the step that removes the most words overall (about half the
total word count goes away). The number of *unique* words only drops a little though,
because the stop-word list only contains around 570 words — everything else is untouched.


### 1.5.1 Turning words into their base form (lemmatisation)

Lemmatisation just means turning a word into its dictionary form. For example:
* `products` → `product`
* `loved` → `love`
* `wrinkles` → `wrinkle`
* `moisturisers` → `moisturiser`

**Why we do it now (after stop-word removal, before counting):**
* Doing it **after** stop-word removal saves time — we don't waste effort on words we
  are about to throw away anyway.
* Doing it **before** counting is important — otherwise `skin` and `skins` are counted
  as two different words. They each end up with a smaller count, which could make one
  of them look rare and get removed in the next step.

We use a small trick: once we have looked up the base form for a word once, we remember
it in a Python dictionary. The same word appears thousands of times in the reviews, so
this saves a lot of repeated work and makes the cell much faster.


In [ ]:
lemmatizer = WordNetLemmatizer()

# A small dictionary that remembers the base form for each word we have already looked up.
_lemma_cache: dict[str, str] = {}

def lemmatise(word: str) -> str:
    """Return the base form of ``word``. Remembers earlier results to save time."""
    if word not in _lemma_cache:
        _lemma_cache[word] = lemmatizer.lemmatize(word)
    return _lemma_cache[word]

# Replace every word with its base form.
tk_reviews = [[lemmatise(w) for w in review] for review in tk_reviews]

stats_print(tk_reviews)


In [ ]:
# A quick check: print a few words that actually changed when we lemmatised, so we
# can see the lemmatiser is doing what we expect.
changed_examples = [
    (w, l) for w, l in list(_lemma_cache.items())[:2000] if w != l
][:15]
print('Examples of words that were changed (original -> base form):')
for w, l in changed_examples:
    print(f'  {w:>15s}  ->  {l}')


**What this did:** The total number of words is exactly the same (each word is just
rewritten, not removed). But the number of *unique* words drops because plurals and
different verb endings now collapse into a single form.


### 1.6 Removing words that appear only once (step 6)

If a word shows up only one time in **all ~ 61,000 reviews put together**, it is almost
always a typo, a brand name nobody else mentions, or some other one-off oddity. A
machine-learning model can't learn anything useful from a word it has only seen once,
and these words make the unique-word list much bigger than it needs to be. So we drop
them.

Note that here we count the **total** number of times a word appears across everything
(the brief calls this *term frequency*). The next step uses a different counting rule.


In [ ]:
# Count how often each word appears across all the reviews put together.
term_freq = Counter(chain.from_iterable(tk_reviews))

# Pick out the words that appear exactly once.
rare_words = {w for w, c in term_freq.items() if c == 1}
print(f'Words that appear only once: {len(rare_words):,}')
print(f'That is about {len(rare_words) / len(term_freq):.1%} of the unique words right now.')

# Drop those words from every review.
tk_reviews = [[w for w in review if w not in rare_words] for review in tk_reviews]

stats_print(tk_reviews)


### 1.7 Removing the 20 most common words across reviews (step 7)

This step uses a slightly different way of counting. Instead of counting the **total**
number of times each word appears, we count the number of **separate reviews** the word
shows up in. Each review can only add 1 to the count, no matter how many times the word
is used in that review. This is called *document frequency*.

The 20 words with the highest document frequency are basically domain-specific stop
words — words that show up in almost every cosmetics review (`good`, `product`, `skin`,
…). They don't help us tell positive reviews from negative ones, so we remove them.


In [ ]:
# Count document frequency: for each review, count each word at most once by
# wrapping the review in set() before passing it to the Counter.
doc_freq = Counter()
for review in tk_reviews:
    doc_freq.update(set(review))

# Pull out the 20 words that show up in the most separate reviews.
top20 = doc_freq.most_common(20)

# Print them as a small table so it's easy to read.
print('Top 20 most common words across reviews (these will be removed):')
print(f'  {"rank":>4s}  {"word":>15s}  {"reviews":>10s}')
for rank, (w, c) in enumerate(top20, start=1):
    print(f'  {rank:>4d}  {w:>15s}  {c:>10,d}')

# Remove them from every review.
top20_words = {w for w, _ in top20}
tk_reviews = [[w for w in review if w not in top20_words] for review in tk_reviews]

stats_print(tk_reviews)


**What this did:** The list above is exactly the kind of words a person would expect to
be in a cosmetics review (`good`, `product`, `shade`, `love`, `skin`, …). Removing them
leaves the more interesting words behind — the ones that actually distinguish one
review from another, which is what we want for the machine-learning models in Task 2
and Task 3.

## 2. Saving the required output files

The brief asks for two files. We create them in this section, following the file-name
and format rules exactly so the marker can compare our files against the expected ones.


### 2.1 Saving `processed.csv`

We keep all the original columns (so Task 3 can still use them). We only change the `review_text` column — we replace
the original text with the cleaned words joined by single spaces.

We pass `index=False` so pandas does not add an extra index column to the CSV.


In [ ]:
from pathlib import Path

# Always save outputs to the project outputs folder
OUT_DIR = Path("../outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

processed_df = raw_df.copy()
processed_df['review_text'] = [' '.join(review) for review in tk_reviews]

processed_csv_path = OUT_DIR / "processed.csv"
processed_df.to_csv(processed_csv_path, index=False)

print(f"Saved processed.csv with {len(processed_df):,} rows and {processed_df.shape[1]} columns.")
print("Path:", processed_csv_path.resolve())
print("Exists:", processed_csv_path.exists())

processed_df[['review_id', 'review_title', 'review_text', 'is_a_buyer']].head()

### 2.2 Saving `vocab.txt`

The brief is very specific about the format of this file:
* One word per line.
* Each line looks like `word:number`.
* Words are sorted in **alphabetical** order.
* The numbers start from **0** and go up by 1 each line.

We build the word list directly from the cleaned reviews, so the words in `vocab.txt`
match the words in `processed.csv` exactly.


In [ ]:
from pathlib import Path

OUT_DIR = Path("../outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

vocab = sorted(set(chain.from_iterable(tk_reviews)))
print(f'Final number of unique words: {len(vocab):,}')

vocab_path = OUT_DIR / "vocab.txt"
with open(vocab_path, 'w', encoding='utf-8') as f:
    f.write('\n'.join(f'{w}:{i}' for i, w in enumerate(vocab)))

# Read back from the same file in outputs/
with open(vocab_path, 'r', encoding='utf-8') as f:
    lines = f.read().splitlines()

print("Path:", vocab_path.resolve())
print("Exists:", vocab_path.exists())
print(f'Number of lines in vocab.txt: {len(lines):,}')
print('First 10 lines:')
for line in lines[:10]:
    print(f'  {line}')
print('Last 5 lines:')
for line in lines[-5:]:
    print(f'  {line}')

**Quick check:** the first line ends in `:0`, the last line ends in `:` followed by
(number of words − 1), the words are all lowercase and in dictionary order. The file
matches the example shown in **Fig. 1** of the brief.


## Summary

We built a cleaning pipeline for the cosmetics and beauty reviews. The steps, in order,
were:

1. Loaded the CSV file and looked only at the `review_text` column.
2. Split each review into words using the pattern given in the brief.
3. Made every word lowercase (done together with step 2 to save time).
4. Removed words shorter than 2 letters.
5. Removed stop words using the supplied `stopwords_en.txt` file.
6. Lemmatised the remaining words (turned each one into its base form). We did this
   step before counting frequencies so that different forms of the same word would be
   counted together.
7. Removed words that appeared only once across all reviews.
8. Removed the 20 words that appeared in the most separate reviews — these turned out
   to be domain-specific stop words like `good`, `product`, `skin` and `love`.

The two files we produced are:
* **`processed.csv`** — the same dataset with the review text replaced by the cleaned
  words. All other columns are kept the same so Task 2 and Task 3 can use them.
* **`vocab.txt`** — an alphabetically sorted list of every unique cleaned word, with a
  number next to each, in the format the brief asks for.

These two files are exactly what Task 2 needs as input.
